# Shallow Profiler


#### Build commands


- `cd ~/argosy`
- `jupyter-book build .`
- `ghp-import -n -p -f _build/html`


[Jupyter Book "Argosy"](https://robfatland.github.io/argosy)


This project near-term goal is described in `~/argosy/ooiproject.md` 


Re-engaging after a time gap: Here are meta-tasks to help organize / present this material

- The `~/argosy/AIPrompt.md` file is the definitive summary/narrative
- Open 2 or 3 Ubuntu `bash` shells including 1 to start Jupyter lab
- Open VS Code with the CA enabled
    - The CA is directed to write either cell code (Jupyter) or standalone
    - The standalone is a different Python distribution so installs are via `pip` in PowerShell
- Establish that the CA can see the WSL file system e.g. by means of the `wsl` utility command in PowerShell
- Develop a presentation deck, link to that
- Reconcile `argosy`, `epipelargosy`, `oceanography` and any other relevant GitHub repos
- Reconcile both Dev machines
- For WSL work: `activate` appropriate Python environments
- Mount the S3 bucket as a local file folder using the alias from `.bash_aliases`

In [1]:
from math import cos, pi

def OceanScienceCalculation():
    '''
    The following is an estimate of DOC in the ocean.
    DOC Dissolved organic carbon in the ocean is ~1000 GTons. Inorganic carbon: 38,000 GTons.
    Near the ocean surface: 100 - 500 micromoles of carbon per kg seawater. The concentration 
    decreases with depth so for the entire water column we can use an attenuation factor 
    of (1/7.5). 
    '''
    radius_earth_meters      = 6378000
    pi                       = 3.141592654
    ocean_percent            = 71
    ocean_mean_depth_meters  = 3700
    water_kg_per_m3          = 1000
    ocean_volume_m3          = 4. * pi * radius_earth_meters**2 * (ocean_percent / 100.) * ocean_mean_depth_meters
    GTons_per_kg             = 1e-12
    ocean_mass_kg            = ocean_volume_m3 * water_kg_per_m3
    ocean_mass_GTons         = ocean_mass_kg * GTons_per_kg

    carbon_surface_um_per_kg = 300
    micromoles_per_mole      = 1e6
    moles_per_micromole      = 1/micromoles_per_mole
    depth_attenuation        = 1/7.5
    carbon_moles_per_kg      = carbon_surface_um_per_kg * moles_per_micromole * depth_attenuation
    carbon_gm_per_mole       = 12
    gm_per_kg                = 1000
    kg_per_gm                = 1 / gm_per_kg
    
    carbon_kg_per_kg         = carbon_moles_per_kg * carbon_gm_per_mole * kg_per_gm
    carbon_in_the_ocean_kg   = ocean_mass_kg * carbon_kg_per_kg
    carbon_in_ocean_GTons    = GTons_per_kg * carbon_in_the_ocean_kg
    
    print("Mass of earth's oceans:", '{:0.2e}'.format(ocean_mass_GTons), 'GTons')
    print("Organic carbon (kg) dissolved per kg of seawater:", round(carbon_kg_per_kg, 12))
    print("Dissolved organic carbon mass, earth's oceans:", round(carbon_in_ocean_GTons, 1), 'GTons')
    print()


# The Oregon coast has a longitude of approximately 124 degrees west
# The three shallow profiler sites are located some distance further west.
def OffshoreDistanceFromNewportOregon(lon):
    '''
    Regional Cabled Array (RCA)-specific calculation. Returns the distance (km) 
    of some site by longitude relative to Newport on the Oregon coast. The
    hardcoded reference location is 44.6 deg north, -124 deg west.
    '''
    ref_lat, ref_lon = 44.6*pi/180, -124*pi/180
    re               = 6378.     # earth radius, kilometers
    return abs(lon*pi/180 - ref_lon)*cos(ref_lat)*re



### Goals, constraints


Goal: Characterize the upper 200 meters of the water column (the photic / epipelagic zone) nine
times per day at three sites: sensors described below. Observing interval is
2015 to present.


The scope of this data analysis procedure is constrained by...
- ...shallow profiler data only; no platform data and no data from other RCA sites
- ...ascent only: Optimize undisturbed water
    - Exception: Some sensors only operate on descent (pCO2; nitrate is ascent I believe)
- ...three RCA shallow profiler sites (with abbreviations)
    - Slope base (sb): Base of the continental shelf
    - Oregon offshore (oo): On the continental shelf
    - Axial base (ab): At Axial seamount
- ...full-resolution datasets as source
    - Data supplied by OOINET
    - Resampling strategies as described below
- ...dual dimension strategy: time dimension for profiles, depth dimension for an individual profile 


Ascent and descent interval metadata is available from GitHub.
    - Wendi Ruef (RCA): `https://github.com/OOI-CabledArray/profileIndices`

## OOI data access


See Reserve notebook


## Sensor types


### Shallow profiler "science pod"


- CTD
- 3 Wavelength Fluorometer
- Dissolved Oxygen
- Nitrate
- Photosynthetically Available Radiation
- Seawater pH
- Single Point Velocity Meter
- Spectral Irradiance
- Spectrophotometer
- pCO2 Water


### Profiler Platform (200m)


These sensors are not the immediate concern.


- 2-Wavelength Fluorometer
- 5-Beam Velocity Profiler (500kHz)
- 5-Beam Velocity Profiler (600kHz)
- Broadband Acoustic Receiver (Hydrophone)
- CTD
- Digital Still Camera
- Dissolved Oxygen
- Seawater pH
- Velocity Profiler (150kHz)

```
import xarray as xr

ds = xr.open_dataset('~/argosy/tmpdata/parexample.nc')
ds

# instruments = ['ctd', 'dissolvedoxygen', 'nitrate']
# instruments = ['nitratedark', 'fluorometer', 'pco2', 'ph']
instruments = ['par']
# instruments = []

for instrument in instruments: 
    ds = xr.open_dataset('~/argosy/tmpdata/' + instrument + 'example.nc')
    print('\n\n' + instrument)
    print('Dimensions:')
    for dim, size in ds.dims.items():
        print(f'  {dim}: {size}')
    print('\nCoordinates:')
    for coord in ds.coords:
        print(coord)
    print('\nVariables:')
    for var in ds.data_vars:
        print(f'  {var}')
    print('\nAttributes')
    for attr in ds.attrs:
        print(f'  {attr}')
    print('\n'*5)

Here is some CTD exploratory code

```
fnm = 'some_ctd_file.nc'
fpath = '~/some/path/to/ctd/'
f = fpath + fnm

ds = xr.open_dataset(f)
# ds produces Dimension: row (9.4M), Coords: nada, Data variables: time, salinity, qc, z
#   and Attributes: 44 in number. Let's move time to Coordinate / Dimension

ds = ds.swap_dims({'obs':'time'})
ds.time[-10:-1]
```

```
import xarray as xr
import pandas as pd

# Load profile times
profiles = pd.read_csv('/home/rob/profileIndices/RS01SBPS_profiles_2018.csv')
profiles['start'] = pd.to_datetime(profiles['start'])
profiles['peak'] = pd.to_datetime(profiles['peak'])

# Load CTD data
ds = xr.open_dataset('/home/rob/argosy/tmpdata/ctdexample.nc')

print('CTD dataset time range:')
print(f'  Min: {ds.time.min().values}')
print(f'  Max: {ds.time.max().values}')

print('\nFirst 4 profile times from CSV:')
for i in range(4):
    print(f'  Profile {profiles.iloc[i]["profile"]}: {profiles.iloc[i]["start"]} to {profiles.iloc[i]["peak"]}')

print('\nDepth range:')
print(f'  Min: {ds.depth.min().values}')
print(f'  Max: {ds.depth.max().values}')

print('\nTemperature range:')
print(f'  Min: {ds.sea_water_temperature.min().values}')
print(f'  Max: {ds.sea_water_temperature.max().values}')

# Check if first profile overlaps with data
profile = profiles.iloc[0]
mask = (ds.time >= profile['start']) & (ds.time <= profile['peak'])
print(f'\nData points in first profile: {mask.sum().values}')

```
import xarray as xr
import pandas as pd

# Load profile times
profiles = pd.read_csv('/home/rob/profileIndices/RS01SBPS_profiles_2018.csv')
profiles['start'] = pd.to_datetime(profiles['start'])
profiles['peak'] = pd.to_datetime(profiles['peak'])

# Load CTD data
ds = xr.open_dataset('/home/rob/argosy/tmpdata/ctdexample.nc')

ctd_start = pd.to_datetime(ds.time.min().values)
ctd_end = pd.to_datetime(ds.time.max().values)

print(f'CTD data range: {ctd_start} to {ctd_end}')

# Find profiles that overlap with CTD data
matching = profiles[(profiles['start'] >= ctd_start) & (profiles['peak'] <= ctd_end)]

print(f'\nFound {len(matching)} matching profiles')
print('\nFirst 4 matching profiles:')
for i in range(min(4, len(matching))):
    row = matching.iloc[i]
    print(f'  Profile {row["profile"]}: {row["start"]} to {row["peak"]}')

In [2]:
# Program "TemperatureChart"

import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

# Load profile times and text > datetimes for columns 2, 3, 4
profiles = pd.read_csv('~/profileIndices/RS01SBPS_profiles_2018.csv')
profiles['start'] = pd.to_datetime(profiles['start'])
profiles['peak'] = pd.to_datetime(profiles['peak'])
profiles['end'] = pd.to_datetime(profiles['end'])

# Load CTD data
ds = xr.open_dataset('/home/rob/tmpdata/ctdexample.nc')

ds = ds.swap_dims({'obs':'time'})

# Find profiles that overlap with CTD data
ctd_start = pd.to_datetime(ds.time.min().values)
ctd_end = pd.to_datetime(ds.time.max().values)
matching = profiles[(profiles['start'] >= ctd_start) & (profiles['peak'] <= ctd_end)]

# Select 4 consecutive profiles
colors = ['red', 'orange', 'green', 'blue']
fig, ax = plt.subplots(figsize=(8, 5))

for i in range(4):
    profile = matching.iloc[i]
    start_time = profile['start']
    peak_time = profile['peak']
    
    # Select data for this profile (ascent: start to peak)
    mask = (ds.time >= start_time) & (ds.time <= peak_time)
    profile_data = ds.where(mask, drop=True)
    
    # Depth is positive
    depth = profile_data['depth']
    
    # Plot temperature vs depth
    ax.plot(profile_data['sea_water_temperature'], depth, color=colors[i])

ax.set_ylim(200, 0)
ax.set_xlim(6, 16)
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Depth (m)')
ax.set_title('CTD Temperature Profiles - Slope Base, 2018')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/home/rob/ctd_temperature_profiles.png', dpi=150)
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/home/rob/profileIndices/RS01SBPS_profiles_2018.csv'

In [3]:
ds.data_vars.keys()

KeysView(Data variables:
    sea_water_pressure_qc_results                      (time) uint8 ...
    sea_water_pressure                                 (time) float64 ...
    sea_water_electrical_conductivity_qartod_results   (time) uint8 ...
    corrected_dissolved_oxygen                         (time) float64 ...
    sea_water_pressure_qc_executed                     (time) uint8 ...
    sea_water_practical_salinity_qc_executed           (time) uint8 ...
    driver_timestamp                                   (time) datetime64[ns] ...
    id                                                 (time) |S36 ...
    conductivity                                       (time) float64 ...
    temperature                                        (time) float64 ...
    sea_water_temperature_qartod_results               (time) uint8 ...
    corrected_dissolved_oxygen_qc_executed             (time) uint8 ...
    corrected_dissolved_oxygen_qc_results              (time) uint8 ...
    pressure_temp      